In [ ]:
from dotenv import load_dotenv
from rich import print as rprint
from langchain.agents import create_agent

load_dotenv()

# 1.没有记忆时

In [ ]:
agent = create_agent(model="deepseek-chat")

In [ ]:
from langchain.messages import HumanMessage

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我叫艾海鹏，我最喜欢仓鼠。")]}
)
rprint(response)

In [ ]:
# 第二次调用，询问我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="我的名字叫什么？我最喜欢什么动物?")]}
)
rprint(response)

# 2.短期记忆

添加会话记忆（短期记忆）分为三步：
- 导入并初始化Checkpointer
- 创建Agent，指定Checkpointer
- 调用Agent，指定thread_id


In [ ]:
from dotenv import load_dotenv
from rich import print as rprint
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

load_dotenv()

agent = create_agent(
    "deepseek-chat",
    checkpointer=InMemorySaver(),
    system_prompt="你以海盗的口吻来回答用户问题。"
)

In [ ]:
from uuid import uuid4

uuid = str(uuid4())
print(uuid)

In [ ]:
config = {"configurable": {"thread_id": uuid}}

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我叫艾海鹏，我最喜欢仓鼠。")]},
    config
)

rprint(response)

In [ ]:
# 第二次调用，询问我的信息，这次带上thread_id，唤起记忆
response = agent.invoke(
    {"messages": [HumanMessage(content="我叫什么名字？我最喜欢的动物是什么？")]},
    config 
)

rprint(response)

# 3.持久记忆

这里我们选择使用Sqlite作为存储方案，首先需要按照langgraph-checkpoint-sqlite依赖：
```
uv add langgraph-checkpoint-sqlite
```
接着，按照以下步骤使用：
- 导入依赖
- 初始化checkpointer
- 自动建表
- 创建Agent，指定checkpointer


In [ ]:
from uuid import uuid4
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents import create_agent

# 连接sqlite
connection = sqlite3.connect("notebooks/第2章-LangChain入门/resources/checkpoint.db", check_same_thread=False)
# 初始化checkpointer
checkpointer = SqliteSaver(connection)
# 自动建表
checkpointer.setup()

# 创建agent
agent = create_agent(
    "deepseek-chat",
    checkpointer=checkpointer,
)

In [ ]:
from uuid import uuid4
# 设定thread_id，作为会话标识
uuid = str(uuid4())
print(uuid)

In [ ]:
from langchain.messages import HumanMessage
from rich import print as rprint

# 设定thread_id，作为会话标识
config = {"configurable": {"thread_id": uuid}}

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我叫艾海鹏，我最喜欢仓鼠。")]},
    config  # 调用时添加thread_id
)

rprint(response)

In [ ]:
# 第二次调用，询问我的信息，这次带上thread_id，唤起记忆
response = agent.invoke(
    {"messages": [HumanMessage(content="我叫什么名字？我最喜欢的动物是什么？")]},
    config  # 调用时添加thread_id
)

rprint(response)

In [ ]:
# 清除记忆
checkpointer.delete_thread(uuid)

# 4.记忆管理

当会话历史过长时，可能会超出模型的上下文窗口限制，常见的解决方案有：
- 修剪消息
- 删除消息
- 总结消息摘要

这里我们演示总结消息摘要的方案


**`SummarizationMiddleware` 核心参数说明：**

| 参数 | 作用 | 支持的单位 |
|------|------|-----------|
| `trigger` | **何时触发**总结 | `("messages", N)` 消息数 / `("tokens", N)` token数 / `("fraction", 0.8)` 模型上限百分比 |
| `keep` | 总结后**保留多少**最近消息 | 同上（仅单个值，不支持组合） |

**`trigger` 支持 AND/OR 组合：**

```python
# OR：任一满足即触发
trigger=[("tokens", 3000), ("messages", 100)]

# AND：所有条件同时满足才触发
trigger={"tokens": 4000, "messages": 10}

# 复杂：(条件组1) OR (条件组2)
trigger=[{"tokens": 5000, "messages": 3}, {"tokens": 3000, "messages": 6}]
```

**本例配置含义：** 消息数 ≥ 6 条时触发总结，总结后仅保留最近 1 条消息。其余被压缩为摘要。


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain.messages import HumanMessage

# 初始化checkpointer
checkpointer = InMemorySaver()
# 初始化中间件
summarization_middleware = SummarizationMiddleware(
    model="deepseek-chat",
    trigger=("messages", 6),
    keep=("messages", 1)
)
# 创建agent
agent = create_agent(
    model="deepseek-chat",
    middleware=[summarization_middleware],
    checkpointer=checkpointer,
)

uuid = str(uuid4())

config: RunnableConfig = {"configurable": {"thread_id": uuid}}
# 制造长会话历史
agent.invoke({"messages": [HumanMessage(content="你好，我是艾海鹏.")]}, config)
agent.invoke({"messages": [HumanMessage(content="我最喜欢的运动是跑步")]}, config)
agent.invoke({"messages": [HumanMessage(content="我最喜欢的动物是仓鼠")]}, config)

In [ ]:
response = agent.invoke({"messages": HumanMessage(content="你还记得我吗？")}, config)
rprint(response)

In [ ]:
# 清除记忆
checkpointer.delete_thread(uuid)

# 5.多层记忆体系

## `SummarizationMiddleware` 的定位

它是一种**线性压缩**方案，适合作为多层记忆体系中的**中期记忆层**。

### ✅ 适用场景

| 场景 | 原因 |
|------|------|
| 客服对话 | 单次会话内，问题解决即结束 |
| 代码助手 | 旧轮次压缩后不影响新轮次改代码 |
| 文档问答 | 早期内容压缩为摘要够用 |
| AI 陪聊 | 对话自然渐进，无需精确回溯 |

### ❌ 局限性

| 问题 | 原因 |
|------|------|
| **摘要丢细节** | 具体信息（邮箱、配置）可能被泛化，无法精确回溯 |
| **渐进失真** | 摘要之上再摘要，类似传话游戏，轮次越久越失真 |
| **不支持精确检索** | 无法查询"第 3 轮提到的 Bug 编号" |
| **跨线程不共享** | 绑定 `thread_id`，换个线程又是全新的 |

## 生产级架构：三层记忆

```
┌──────────────────────────────────────────────┐
│  ① 短期记忆 — 滑动窗口                          │
│  └─ 最近 N 轮原文，不压缩（对应 keep 做的事）       │
├──────────────────────────────────────────────┤
│  ② 中期记忆 — SummarizationMiddleware 在这一层  │
│  └─ 超出窗口的消息 → 压缩为摘要                  │
├──────────────────────────────────────────────┤
│  ③ 长期记忆 — 结构化持久存储                     │
│  └─ 用户画像、偏好、关键事实 → Store / 向量库      │
└──────────────────────────────────────────────┘
```

**`SummarizationMiddleware` 是完整记忆体系中的一环，而非唯一方案。** 生产级 AI 助手还需补充滑动窗口 + 关键信息提取 + RAG 检索，才能做到精确、持久、可检索的长期记忆。
